In [1]:
# ============================================================
# CONFIGURAZIONE — modifica solo questa sezione
# ============================================================
SEASON = "2025-2026"

# ============================================================
# Setup: imports, ambiente, path
# ============================================================
import os, sys
import pandas as pd

# Aggiunge la root del repo al path (per importare config/)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Mount Drive se siamo su Colab
def is_colab():
    return "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if is_colab():
    from google.colab import drive
    drive.mount('/content/drive')

from config.paths import build_file_list, build_base_path

# Carica la lista file e gli avversari dalla config
FILES, avversari = build_file_list(season=SEASON)

# base_path per output PDF
base_path = build_base_path(season=SEASON)

print(f"Stagione: {SEASON}")
print(f"Partite da caricare: {len(FILES)}")


Stagione: 2025-2026
Partite da caricare: 28


In [2]:
all_matches_df = []

for i, file_path in enumerate(FILES):
    try:
        # Read the Excel file into a DataFrame
        df = pd.read_excel(file_path, skiprows=1)
        df.dropna(subset=['Numero'], inplace=True)
        df['Numero'] = df['Numero'].astype(int)

        # Add the 'match' column with the corresponding opponent name
        df['match'] = avversari[i]

        # Append the DataFrame to the list
        all_matches_df.append(df)
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")

# Concatenate all DataFrames into a single one
final_df = pd.concat(all_matches_df, ignore_index=True)

print(f"Caricati {len(all_matches_df)} match")
# Display the first few rows of the final DataFrame
display(final_df.head())

# 1. Normalize 'Cognome' (first letter uppercase, rest lowercase)
final_df['Cognome'] = final_df['Cognome'].astype(str).apply(lambda x: x.replace('’', "'").strip().capitalize())

# 2. Correct player numbers: update 'Numero' based on the last encountered 'Numero' for each 'Cognome'
# Create a mapping of Cognome to its last encountered Numero
cognome_to_last_numero = {}
for index, row in final_df.iterrows():
    cognome_to_last_numero[row['Cognome']] = row['Numero']

# Apply this mapping to update the 'Numero' column
final_df['Numero'] = final_df['Cognome'].map(cognome_to_last_numero)

display(final_df.head())

Caricati 28 match


,id,Tipo,Voto,Cognome,Numero,Posizione Giocatore,Posizione Palleggiatore,Numero Set,Punti Locali,Punti Ospiti,match
0,0,battuta,-,Cepparano,11,p1,p3,1,0,0,A-Lazio
1,1,muro,=,Pessei,13,p3,p3,1,0,0,A-Lazio
2,2,ricezione,+,Pedico,31,p5,p3,1,0,1,A-Lazio
3,3,attacco,#,Chimenton,4,p4,p3,1,0,1,A-Lazio
4,4,battuta,-,Pessei,13,p1,p2,1,1,1,A-Lazio


,id,Tipo,Voto,Cognome,Numero,Posizione Giocatore,Posizione Palleggiatore,Numero Set,Punti Locali,Punti Ospiti,match
0,0,battuta,-,Cepparano,11,p1,p3,1,0,0,A-Lazio
1,1,muro,=,Pessei,13,p3,p3,1,0,0,A-Lazio
2,2,ricezione,+,Pedico,31,p5,p3,1,0,1,A-Lazio
3,3,attacco,#,Chimenton,4,p4,p3,1,0,1,A-Lazio
4,4,battuta,-,Pessei,13,p1,p2,1,1,1,A-Lazio


In [3]:
# ============================================================
# CHECK TERNE GIOCATORI
# Rileva giocatori non ancora in player_identities.csv
# e chiede come associarli prima di procedere.
# ============================================================
import csv
from pathlib import Path

PLAYERS_CSV     = Path(REPO_ROOT) / 'config' / 'players.csv'
IDENTITIES_CSV  = Path(REPO_ROOT) / 'config' / 'player_identities.csv'

def load_identities():
    identities = []
    try:
        with open(IDENTITIES_CSV, newline='') as f:
            reader = csv.DictReader(f)
            identities = list(reader)
    except FileNotFoundError:
        pass
    return identities

def load_players():
    players = {}
    try:
        with open(PLAYERS_CSV, newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                players[row['player_id']] = row['cognome_canonical']
    except FileNotFoundError:
        pass
    return players

def next_player_id(players):
    if not players:
        return 'P001'
    nums = [int(pid[1:]) for pid in players if pid.startswith('P')]
    return f'P{max(nums)+1:03d}'

def save_identity(player_id, team, cognome, numero):
    with open(IDENTITIES_CSV, 'a', newline='') as f:
        writer = csv.writer(f, lineterminator='\n')
        writer.writerow([player_id, team, cognome, numero])

def save_player(player_id, cognome):
    with open(PLAYERS_CSV, 'a', newline='') as f:
        writer = csv.writer(f, lineterminator='\n')
        writer.writerow([player_id, cognome])

identities = load_identities()
players    = load_players()

known = {(r['team'], r['cognome'], r['numero']): r['player_id'] for r in identities}

team = 'Decimo'
unresolved = []

for _, row in final_df.iterrows():
    cognome = str(row['Cognome'])
    numero  = str(int(row['Numero']))
    key     = (team, cognome, numero)
    if key not in known:
        unresolved.append(key)

unresolved = list(dict.fromkeys(unresolved))  # dedup mantenendo ordine

if not unresolved:
    print("✓ Tutte le terne riconosciute. Nessuna azione richiesta.")
else:
    print(f"⚠ {len(unresolved)} terna/e non riconosciuta/e:\n")
    for (t, cog, num) in unresolved:
        print(f"  Team: {t} | Cognome: {cog} | Numero: {num}")
        
        # Cerca candidati: stesso cognome in qualsiasi stagione (esclusi gli IGNORED)
        candidates = [r for r in identities if r['cognome'].lower() == cog.lower() and r['player_id'] != 'IGNORED']
        
        print("  Candidati trovati:")
        for idx, c in enumerate(candidates, 1):
            print(f"    [{idx}] {c['player_id']} · {c['cognome']} · {c['team']} · nr. {c['numero']}")
        print(f"    [{len(candidates)+1}] Nuovo giocatore")
        print(f"    [{len(candidates)+2}] Ignora definitivamente (escluso dalle classifiche, non richiesto più)")
        
        scelta = input("  Scelta: ").strip()
        try:
            scelta_int = int(scelta)
        except ValueError:
            scelta_int = len(candidates) + 2

        if 1 <= scelta_int <= len(candidates):
            pid = candidates[scelta_int - 1]['player_id']
            save_identity(pid, t, cog, num)
            known[(t, cog, num)] = pid
            print(f"  → Associato a {pid}\n")
        elif scelta_int == len(candidates) + 1:
            pid = next_player_id(players)
            save_player(pid, cog)
            save_identity(pid, t, cog, num)
            players[pid] = cog
            known[(t, cog, num)] = pid
            print(f"  → Nuovo giocatore creato: {pid}\n")
        else:
            # Persistito come IGNORED: la terna sarà riconosciuta nei run futuri
            # (nessun nuovo prompt) ma resterà esclusa dalle classifiche (vedi cella FILTRO).
            save_identity('IGNORED', t, cog, num)
            known[(t, cog, num)] = 'IGNORED'
            print("  → Ignorato definitivamente.\n")

    print("✓ Check completato. Puoi procedere.")


✓ Tutte le terne riconosciute. Nessuna azione richiesta.


In [4]:

# ============================================================
# FILTRO — esclude giocatori non in player_identities.csv
# (e le terne marcate player_id == 'IGNORED', ignorate definitivamente)
# ============================================================
identities_df = pd.read_csv(IDENTITIES_CSV)
identities_df = identities_df[identities_df['player_id'] != 'IGNORED']

# costruisce le chiavi riconosciute
known_keys = set(
    zip(
        identities_df['team'],
        identities_df['cognome'],
        identities_df['numero'].astype(str)
    )
)

# filtra final_df
before = len(final_df)
final_df = final_df[
    final_df.apply(
        lambda r: ('Decimo', r['Cognome'], str(int(r['Numero']))) in known_keys,
        axis=1
    )
]
after = len(final_df)

print(f"Righe prima del filtro: {before}")
print(f"Righe dopo il filtro:   {after}")
print(f"Escluse: {before - after}")

Righe prima del filtro: 10842
Righe dopo il filtro:   10572
Escluse: 270


# Classifiche

In [5]:

# Funzioni core delle classifiche estratte in src/rankings.py (riuso cross-notebook / cross-stagione).
from src.rankings import (
    calculate_player_scores_by_type,
    build_ace_man,
    build_best_attack,
    build_spike_guarantee,
    build_best_block,
    build_best_rec,
    build_srv_shame,
)


# -------------------------
# Controlli rapidi base
# -------------------------
battuta_scores = calculate_player_scores_by_type(final_df, 'battuta')
print("Risultati per 'battuta':")
display(battuta_scores.head())

attacco_scores_debug = calculate_player_scores_by_type(final_df, 'attacco')
print("\nRisultati per 'attacco':")
display(attacco_scores_debug.head())

attacco_scores_filtered = calculate_player_scores_by_type(
    final_df,
    'attacco',
    ignore_list=['Chimenton', 'Cepparano']
)
print("\nRisultati per 'attacco' (escludendo Chimenton e Cepparano):")
display(attacco_scores_filtered.head())


# -------------------------
# 1) ACE MAN
# - Aggiunge %# = ace / battute totali
# - Entra solo chi ha almeno 10 battute
# - Ordina per ace assoluti, poi per %#
# -------------------------
srv_scores = calculate_player_scores_by_type(final_df, 'battuta', ignore_list=['Pedico', 'Amore'])
ace_man = build_ace_man(srv_scores, min_battute=10)
display(ace_man)


# -------------------------
# 2) SPIKE LEADER
# - Classifica per numero assoluto di attacchi punto '#'
# -------------------------
attacco_scores = calculate_player_scores_by_type(final_df, 'attacco', ignore_list=['Pedico'])
best_attack = build_best_attack(attacco_scores)
display(best_attack)


# -------------------------
# 3) SPIKE GUARANTEE
# - Eff% = (# - (/ + =)) / totale attacchi
# - Entra solo chi ha almeno 50 attacchi
# - Ordina per Eff%
# -------------------------
spike_guarantee = build_spike_guarantee(attacco_scores, min_attacchi=50)
display(spike_guarantee)


# -------------------------
# 4) BLOCK MONSTER
# - Classifica per muri punto '#'
# -------------------------
blk_scores = calculate_player_scores_by_type(final_df, 'muro', ignore_list=['Pedico'])
best_block = build_best_block(blk_scores)
display(best_block)


# -------------------------
# 5) MIGLIOR RICEVITORE
# - Pos% = (# + +) / ricezioni totali
# - Prf% = # / ricezioni totali
# - Entra solo chi ha almeno 40 ricezioni
# - Ordina per Pos%
# -------------------------
pos_rec = calculate_player_scores_by_type(
    final_df,
    'ricezione',
    ignore_list=['Amore', 'Caranzetti']
)
best_rec = build_best_rec(pos_rec, min_ricezioni=40)
display(best_rec)


# -------------------------
# 6) MURO DELLA VERGOGNA / BATTUTE SBAGLIATE
# - Aggiunge Err% = errori '=' / battute totali
# - Ordina principalmente per numero assoluto di errori '='
# -------------------------
srv_shame = build_srv_shame(srv_scores)
display(srv_shame)


Risultati per 'battuta':


Voto,#,+,!,/,-,=,Totale
Giocatore,,,,,,,
Caranzetti 14,13,47,39,18,127,30,274
Carrer 18,11,27,27,7,65,38,175
Cepparano 11,23,41,45,17,138,51,315
Chimenton 4,24,59,58,12,186,43,382
D'arienzo 22,18,33,27,9,76,57,220



Risultati per 'attacco':


Voto,#,+,!,/,-,=,Totale
Giocatore,,,,,,,
Caranzetti 14,32,13,14,7,39,5,110
Carrer 18,109,31,16,17,86,46,305
Cepparano 11,176,52,57,34,223,85,627
Chimenton 4,361,138,75,46,443,77,1140
D'arienzo 22,186,63,34,34,128,81,526



Risultati per 'attacco' (escludendo Chimenton e Cepparano):


...

Voto,Pos,Giocatore,#
0,1,Chimenton 4,361
1,2,D'arienzo 22,186
2,3,Cepparano 11,176
3,4,Pessei 13,146
4,5,Carrer 18,109
5,6,Moscetta 30,60
6,7,Sardella 66,47
7,8,Caranzetti 14,32
8,9,Licenziati 1,7
9,10,Liberatori 3,4


Voto,Pos,Giocatore,Eff%,Attacchi
0,1,Pessei 13,32%,309
1,2,Chimenton 4,21%,1140
2,3,Caranzetti 14,18%,110
3,4,Carrer 18,15%,305
4,5,Moscetta 30,15%,176
5,6,D'arienzo 22,13%,526
6,7,Cepparano 11,9%,627
7,8,Sardella 66,1%,175


Voto,Pos,Giocatore,#
0,1,Moscetta 30,38
1,2,Pessei 13,37
2,3,Carrer 18,29
3,4,Chimenton 4,27
4,5,D'arienzo 22,20
5,6,Caranzetti 14,14
6,7,Cepparano 11,11
7,8,Liberatori 3,1
8,9,Sardella 66,1
9,10,Licenziati 1,0


Voto,Pos,Giocatore,Pos%,Ricezioni,Prf%
0,1,Chimenton 4,58%,641,36%
1,2,Pedico 31,54%,721,36%
2,3,Cepparano 11,54%,435,28%
3,4,Sardella 66,52%,54,24%


Voto,Pos,Giocatore,=,Battute,Err%
0,1,D'arienzo 22,57,220,26%
1,2,Cepparano 11,51,315,16%
2,3,Chimenton 4,43,382,11%
3,4,Pessei 13,39,286,14%
4,5,Carrer 18,38,175,22%
5,6,Caranzetti 14,30,274,11%
6,7,Sardella 66,16,90,18%
7,8,Moscetta 30,15,302,5%
8,9,Licenziati 1,12,118,10%
9,10,Liberatori 3,2,5,40%


# PDF

In [6]:

ace_man_formatted = ace_man.to_string(index=False)
best_attack_formatted = best_attack.to_string(index=False)
spike_guarantee_formatted = spike_guarantee.to_string(index=False)
best_block_formatted = best_block.to_string(index=False)
best_rec_formatted = best_rec.to_string(index=False)
srv_shame_formatted = srv_shame.to_string(index=False)

print("Formatted ace_man:")
print(ace_man_formatted)
print("\nFormatted best_attack:")
print(best_attack_formatted)
print("\nFormatted spike_guarantee:")
print(spike_guarantee_formatted)
print("\nFormatted best_block:")
print(best_block_formatted)
print("\nFormatted best_rec:")
print(best_rec_formatted)
print("\nFormatted srv_shame:")
print(srv_shame_formatted)

try:
    from fpdf import FPDF
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'fpdf2'])
    from fpdf import FPDF

import pandas as pd
import os

# Create instance of FPDF class with landscape orientation.
pdf = FPDF('L', 'mm', 'A4')

# Add a page.
pdf.add_page()
pdf.set_text_color(0, 0, 0)
pdf.set_fill_color(255, 255, 255)


def add_ranking_to_pdf(pdf_obj, title, df_data, title_color=(0, 0, 0),
                       x=None, y=None, table_width=None):
    """Aggiunge una classifica come tabella nel PDF."""
    if x is None:
        x = pdf_obj.get_x()
    if y is None:
        y = pdf_obj.get_y()

    pdf_obj.set_xy(x, y)
    pdf_obj.set_font('Arial', '', 10)

    headers = df_data.columns.tolist()
    data_rows = df_data.values.tolist()

    col_widths = []
    pos_col_width = 10
    player_col_index = -1

    for i, col in enumerate(headers):
        if col == 'Pos':
            col_widths.append(pos_col_width)
        elif col == 'Giocatore':
            player_col_index = i
            col_widths.append(0)
        else:
            max_width = pdf_obj.get_string_width(str(col)) + 6
            for row in data_rows:
                cell_content_width = pdf_obj.get_string_width(str(row[i])) + 6
                if cell_content_width > max_width:
                    max_width = cell_content_width
            col_widths.append(max_width)

    if player_col_index != -1:
        giocatore_max_content_width = pdf_obj.get_string_width(headers[player_col_index]) + 6
        for row in data_rows:
            cell_content_width = pdf_obj.get_string_width(str(row[player_col_index])) + 6
            if cell_content_width > giocatore_max_content_width:
                giocatore_max_content_width = cell_content_width
        if col_widths[player_col_index] == 0:
            col_widths[player_col_index] = giocatore_max_content_width

    current_total_width = sum(col_widths)
    if table_width is not None and current_total_width > table_width:
        scale_factor = table_width / current_total_width
        col_widths = [w * scale_factor for w in col_widths]
        current_total_width = table_width
    elif table_width is None:
        page_width = pdf_obj.w - 2 * pdf_obj.l_margin
        if current_total_width > page_width:
            scale_factor = page_width / current_total_width
            col_widths = [w * scale_factor for w in col_widths]
            current_total_width = sum(col_widths)

    original_font = pdf_obj.font_family
    original_font_style = pdf_obj.font_style
    original_font_size = pdf_obj.font_size

    pdf_obj.set_text_color(*title_color)
    pdf_obj.set_fill_color(200, 220, 255)
    pdf_obj.set_font('Arial', 'B', 12)
    pdf_obj.set_x(x)

    processed_title = title.replace('<br>', '\n')
    pdf_obj.multi_cell(current_total_width, 6, processed_title, 1, 'C', 1)

    pdf_obj.set_text_color(0, 0, 0)
    pdf_obj.set_fill_color(200, 220, 255)
    pdf_obj.set_font(original_font, original_font_style, original_font_size)

    # Header tabella.
    pdf_obj.set_font('Arial', 'B', 9)
    pdf_obj.set_x(x)
    for i, header in enumerate(headers):
        align = 'L' if headers[i] == 'Giocatore' else 'C'
        pdf_obj.cell(col_widths[i], 7, str(header), 1, new_x='RIGHT', new_y='TOP', align=align, fill=1)
    pdf_obj.ln()

    # Righe tabella.
    pdf_obj.set_font('Courier', '', 8)
    fill = False
    for row in data_rows:
        pdf_obj.set_x(x)
        if fill:
            pdf_obj.set_fill_color(240, 240, 240)
        else:
            pdf_obj.set_fill_color(255, 255, 255)

        for i, item in enumerate(row):
            align = 'L' if headers[i] == 'Giocatore' else 'C'
            pdf_obj.cell(col_widths[i], 6, str(item), 1, new_x='RIGHT', new_y='TOP', align=align, fill=fill)
        pdf_obj.ln()
        fill = not fill


# Colori titoli.
color_ace_man = (0, 0, 255)
color_best_attack = (0, 0, 255)
color_spike_guarantee = (0, 0, 255)
color_best_block = (0, 0, 255)
color_best_rec = (0, 0, 255)
color_srv_shame = (255, 0, 0)

# Dimensioni e layout pagina A4 landscape.
margin = 3
table_spacing_x = 4
table_spacing_y = 8

# Larghezze aggiornate per le nuove colonne percentuali.
width_ace_man = 72
width_best_attack = 58
width_spike_guarantee = 74
width_best_block = 58
width_best_rec = 82
width_srv_shame = 76

current_x = margin
current_y = margin

# Riga 1: tre classifiche.
add_ranking_to_pdf(pdf, "Ace Man", ace_man, color_ace_man, x=current_x, y=current_y, table_width=width_ace_man)
current_x += width_ace_man + table_spacing_x

add_ranking_to_pdf(pdf, "Spike Leader", best_attack, color_best_attack, x=current_x, y=current_y, table_width=width_best_attack)
current_x += width_best_attack + table_spacing_x

add_ranking_to_pdf(pdf, "Spike Guarantee", spike_guarantee, color_spike_guarantee, x=current_x, y=current_y, table_width=width_spike_guarantee)
current_x += width_spike_guarantee + table_spacing_x

add_ranking_to_pdf(pdf, "Block Monster", best_block, color_best_block, x=current_x, y=current_y, table_width=width_best_block)

# Riga 2: ricezione + battute sbagliate.
current_x = margin
current_y = pdf.get_y() + table_spacing_y

add_ranking_to_pdf(pdf, "Miglior ricevitore", best_rec, color_best_rec, x=current_x, y=current_y, table_width=width_best_rec)
current_x += width_best_rec + table_spacing_x

add_ranking_to_pdf(pdf, "Muro della vergogna<br>(battute sbagliate)", srv_shame, color_srv_shame, x=current_x, y=current_y, table_width=width_srv_shame)

# Salvataggio PDF.
output_dir = str(base_path / 'classifiche_decimo') + '/'
os.makedirs(output_dir, exist_ok=True)
output_filename = output_dir + f'classifiche ({len(all_matches_df)}).pdf'
pdf.output(output_filename)

print(f"PDF with structured tables saved as {output_filename}")

Formatted ace_man:
 Pos     Giocatore  #  Battute %#
   1   Chimenton 4 24      382 6%
   2  Cepparano 11 23      315 7%
   3  D'arienzo 22 18      220 8%
   4     Pessei 13 18      286 6%
   5   Moscetta 30 17      302 6%
   6 Caranzetti 14 13      274 5%
   7     Carrer 18 11      175 6%
   8   Sardella 66  5       90 6%
   9  Licenziati 1  3      118 3%

Formatted best_attack:
 Pos     Giocatore   #
   1   Chimenton 4 361
   2  D'arienzo 22 186
   3  Cepparano 11 176
   4     Pessei 13 146
   5     Carrer 18 109
   6   Moscetta 30  60
   7   Sardella 66  47
   8 Caranzetti 14  32
   9  Licenziati 1   7
  10  Liberatori 3   4

Formatted spike_guarantee:
 Pos     Giocatore Eff%  Attacchi
   1     Pessei 13  32%       309
   2   Chimenton 4  21%      1140
   3 Caranzetti 14  18%       110
   4     Carrer 18  15%       305
   5   Moscetta 30  15%       176
   6  D'arienzo 22  13%       526
   7  Cepparano 11   9%       627
   8   Sardella 66   1%       175

Formatted best_block:
 Pos   

PDF with structured tables saved as /Users/Marco.Miccheli/Library/CloudStorage/GoogleDrive-marco86sim@gmail.com/Il mio Drive/Pallavolo/Decimo Roma/2025-2026/Serie D/Match analysis/classifiche_decimo/classifiche (28).pdf


/var/folders/zv/mhrw7kq95152xcjbqyhq0gjw0000gp/T/ipykernel_63835/4207456930.py:50: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf_obj.set_font('Arial', '', 10)
/var/folders/zv/mhrw7kq95152xcjbqyhq0gjw0000gp/T/ipykernel_63835/4207456930.py:100: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf_obj.set_font('Arial', 'B', 12)
/var/folders/zv/mhrw7kq95152xcjbqyhq0gjw0000gp/T/ipykernel_63835/4207456930.py:111: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf_obj.set_font('Arial', 'B', 9)
